# SAGE-GI — third foundation model (Nucleotide Transformer v2)

The paper argues that **species-aware** pre-training, not genomic pre-training in
general, is what makes a foundation model useful here. That currently rests on one
matched pair (DNABERT-S vs DNABERT-2).

This notebook adds a second *general-purpose* genomic model as a control:
**Nucleotide Transformer v2 (50M, multi-species)** — masked-language pre-training on
multi-species DNA, no species-aware contrastive stage. If it also underperforms
hand-crafted composition, the claim strengthens considerably.

Same tiling, same pooling, same output format as the main notebook, so it drops
straight into the existing pipeline. ~20–30 min on a T4.

> **Kaggle:** Accelerator → **GPU T4 x2**, Internet → **On**.
> **Colab:** Runtime → Change runtime type → **T4 GPU**.


In [ ]:
import os, subprocess, sys
IN_COLAB  = 'google.colab' in sys.modules or os.path.isdir('/content')
IN_KAGGLE = os.path.isdir('/kaggle')
print('environment:', 'colab' if IN_COLAB else ('kaggle' if IN_KAGGLE else 'local'))
try:
    gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                          '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ''
print('GPU:', gpu or 'NONE DETECTED — turn on the GPU accelerator')


In [ ]:
# Pin transformers, exactly as the DNABERT notebook does. Nucleotide Transformer
# is loaded through `trust_remote_code`, and its bundled `modeling_esm.py` imports
# `find_pruneable_heads_and_indices` and `prune_linear_layer` from
# `transformers.pytorch_utils`. Both were removed in transformers 5, so the
# preinstalled Kaggle/Colab version fails with:
#   ImportError: cannot import name 'find_pruneable_heads_and_indices'
!pip -q install 'transformers==4.46.3' 'tokenizers<0.21' einops 2>&1 | tail -2
import transformers, transformers.pytorch_utils as _pu
print('transformers', transformers.__version__)
assert hasattr(_pu, 'find_pruneable_heads_and_indices'), (
    'this transformers release removed the symbols the model code needs; '
    'restart the kernel so the pinned version is the one imported')
print('required symbols present: OK')


In [ ]:
from pathlib import Path
ROOT   = Path('/kaggle/working/sagegi' if IN_KAGGLE else
              ('/content/sagegi' if IN_COLAB else './sagegi'))
GENOME = ROOT/'genomes'; EMB = ROOT/'emb'
for p in (GENOME, EMB): p.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'
# Pin the revision: the repository ships its own modelling code, and an upstream
# edit would silently change the embeddings between runs.
REVISION = '81b29e5786726d891dbf929404ef20adca5b36f1'
MODEL_TAG = 'NT-v2-50M'      # directory name used downstream
TILE, PCA_K, BATCH = 1000, 96, 64
print(ROOT)


In [ ]:
ACCESSIONS = ["AC_000091.1", "NC_000907.1", "NC_000913.2", "NC_002516.2", "NC_002655.2", "NC_002695.1", "NC_002758.2", "NC_002935.2", "NC_002944.2", "NC_002947.3", "NC_003047.1", "NC_003197.1", "NC_003198.1", "NC_003485.1", "NC_003909.8", "NC_003923.1", "NC_003997.3", "NC_004070.1", "NC_004088.1", "NC_004116.1", "NC_004337.1", "NC_004347.1", "NC_004368.1", "NC_004431.1", "NC_004459.2", "NC_004578.1", "NC_004603.1", "NC_004606.1", "NC_004631.1", "NC_004722.1", "NC_004741.1", "NC_005071.1", "NC_005139.1", "NC_005773.3", "NC_005945.1", "NC_006086.1", "NC_006155.1", "NC_006350.1", "NC_006511.1", "NC_006905.1", "NC_007005.1", "NC_007146.2", "NC_007297.1", "NC_007335.2", "NC_007384.1", "NC_007432.1", "NC_007434.1", "NC_007510.1", "NC_007530.2", "NC_007606.1", "NC_007613.1", "NC_007651.1", "NC_007761.1", "NC_007778.1", "NC_007793.1", "NC_007946.1", "NC_007958.1", "NC_008022.1", "NC_008023.1", "NC_008024.1", "NC_008060.1", "NC_008061.1", "NC_008253.1", "NC_008258.1", "NC_008321.1", "NC_008322.1", "NC_008390.1", "NC_008463.1", "NC_008543.1", "NC_008563.1", "NC_008577.1", "NC_008595.1", "NC_008600.1", "NC_008611.1", "NC_008750.1", "NC_008819.1", "NC_009052.1", "NC_009074.1", "NC_009076.1", "NC_009080.1", "NC_009085.1", "NC_009256.1", "NC_009342.1", "NC_009438.1", "NC_009445.1", "NC_009485.1", "NC_009487.1", "NC_009504.1", "NC_009512.1", "NC_009567.1", "NC_009632.1", "NC_009665.1", "NC_009708.1", "NC_009782.1", "NC_009783.1", "NC_009800.1", "NC_009801.1", "NC_009879.1", "NC_009900.1", "NC_009901.1", "NC_009997.1", "NC_010084.1", "NC_010102.1", "NC_010159.1", "NC_010167.1", "NC_010184.1", "NC_010322.1", "NC_010334.1", "NC_010380.1", "NC_010400.1", "NC_010410.1", "NC_010465.1", "NC_010473.1", "NC_010498.1", "NC_010501.1", "NC_010515.1", "NC_010551.1", "NC_010582.1", "NC_010612.1", "NC_011770.1"]
print(len(ACCESSIONS), 'accessions')


## 1 · Genomes


In [ ]:
import time, threading, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
EUTILS='https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
EMAIL, TOOL = '1024052020@grad.cse.buet.ac.bd', 'SAGE-GI'
_gate,_last = threading.Semaphore(1),[0.0]
def fetch(acc, retries=5):
    url=f'{EUTILS}?db=nuccore&id={acc}&rettype=fasta&retmode=text&tool={TOOL}&email={EMAIL}'
    for a in range(retries):
        try:
            with _gate:
                w=0.4-(time.time()-_last[0])
                if w>0: time.sleep(w)
                _last[0]=time.time()
            with urllib.request.urlopen(url, timeout=180) as r:
                t=r.read().decode('utf-8','replace')
            if t.startswith('>') and len(t)>1000: return t
            raise ValueError('short payload')
        except Exception: time.sleep(3*(a+1))
    raise RuntimeError(acc)
todo=[a for a in ACCESSIONS if not (GENOME/f'{a}.fna').exists()]
print(len(todo),'genomes to fetch')
with ThreadPoolExecutor(3) as ex:
    futs={ex.submit(fetch,a):a for a in todo}
    for i,f in enumerate(as_completed(futs),1):
        (GENOME/f'{futs[f]}.fna').write_text(f.result())
        if i%20==0 or i==len(todo): print(f'  {i}/{len(todo)}', flush=True)
print(len(list(GENOME.glob('*.fna'))),'FASTA ready')


## 2 · Extraction


In [ ]:
import numpy as np, torch, time, gc
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForMaskedLM

def read_fasta(p):
    out=[]
    with open(p) as fh:
        for line in fh:
            if not line.startswith('>'): out.append(line.strip())
    return ''.join(out).upper()

def l2(E):
    E=E.astype(np.float32)
    return E/np.maximum(np.linalg.norm(E,axis=1,keepdims=True),1e-8)

def load(half=True):
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION,
                                        trust_remote_code=True)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_ID, revision=REVISION,
                                             trust_remote_code=True)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if dev.type=='cuda' and half: m = m.half()
    return tok, m.eval().to(dev), dev

tok, model, dev = load(half=True)
print('device', dev, '| dtype', next(model.parameters()).dtype, '| params',
      round(sum(p.numel() for p in model.parameters())/1e6,1), 'M')

@torch.no_grad()
def embed_genome(seq, tile=None, batch=None):
    tile = tile or TILE; batch = batch or BATCH
    starts = np.arange(max(0,len(seq)//tile), dtype=np.int64)*tile
    out=None
    for b0 in range(0,len(starts),batch):
        chunk = starts[b0:b0+batch]
        enc = tok([seq[s:s+tile] for s in chunk], return_tensors='pt',
                  padding=True, truncation=True, max_length=2048)
        ids, mask = enc['input_ids'].to(dev), enc['attention_mask'].to(dev)
        h = model(input_ids=ids, attention_mask=mask,
                  output_hidden_states=True)['hidden_states'][-1]
        mm = mask.unsqueeze(-1).to(h.dtype)
        pooled = ((h*mm).sum(1)/mm.sum(1).clamp(min=1)).float().cpu().numpy()
        if out is None: out = np.empty((len(starts), pooled.shape[1]), np.float16)
        out[b0:b0+len(chunk)] = pooled.astype(np.float16)
    return out, starts

# Half precision is used for speed; verify it does not perturb the embeddings,
# and fall back to fp32 automatically rather than corrupt the run.
USE_HALF = torch.cuda.is_available()
if USE_HALF:
    probe = read_fasta(GENOME/f'{ACCESSIONS[0]}.fna')[:16*TILE]
    e16,_ = embed_genome(probe)
    del model; gc.collect(); torch.cuda.empty_cache()
    tok, model, dev = load(half=False)
    e32,_ = embed_genome(probe)
    a,b = e16.astype(np.float64), e32.astype(np.float64)
    cos = (a*b).sum(1)/(np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1))
    print(f'fp16 vs fp32 cosine: min {cos.min():.6f} mean {cos.mean():.6f}')
    USE_HALF = bool(cos.min() > 0.999)
    if not USE_HALF: print('half precision perturbs embeddings - staying in fp32')
    del model; gc.collect(); torch.cuda.empty_cache()
    tok, model, dev = load(half=USE_HALF)
print('USE_HALF =', USE_HALF, '| dtype', next(model.parameters()).dtype)

outdir = EMB/f'{MODEL_TAG}_t{TILE}'; outdir.mkdir(parents=True, exist_ok=True)
todo = [a for a in ACCESSIONS if not (outdir/f'{a}.npz').exists()]
print(len(todo),'genomes to embed')
t0, done = time.time(), 0
for i, acc in enumerate(todo, 1):
    seq = read_fasta(GENOME/f'{acc}.fna')
    t = time.time()
    emb, starts = embed_genome(seq)
    k = int(min(PCA_K, emb.shape[1], max(2, emb.shape[0]-1)))
    pca = PCA(n_components=k, svd_solver='randomized', random_state=0)
    pcs = pca.fit_transform(l2(emb)).astype(np.float32)
    np.savez_compressed(outdir/f'{acc}.npz', pcs=pcs, starts=starts,
        explained_variance_ratio=pca.explained_variance_ratio_.astype(np.float32),
        tile=TILE, step=TILE, genome_len=len(seq), model=MODEL_TAG)
    dt=time.time()-t; done+=len(seq)
    rate=done/(time.time()-t0)
    left=sum((GENOME/f'{a}.fna').stat().st_size for a in todo[i:])
    print(f'[{i}/{len(todo)}] {acc} {len(seq):>9,} bp {dt:6.1f}s '
          f'{len(seq)/dt:>8,.0f} bp/s  ETA {left/max(rate,1)/60:5.1f} min', flush=True)
print('done:', len(list(outdir.glob('*.npz'))), 'genomes')


## 3 · Export


In [ ]:
import shutil
d = EMB/f'{MODEL_TAG}_t{TILE}'
if IN_KAGGLE:
    a = Path('/kaggle/working')/d.name
    shutil.make_archive(str(a),'zip',root_dir=str(d.parent),base_dir=d.name)
    shutil.rmtree(GENOME, ignore_errors=True)
    print('wrote', a.with_suffix('.zip'),
          f'({a.with_suffix(".zip").stat().st_size/1e6:.0f} MB)')
elif IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/SAGE-GI-embeddings'); dest.mkdir(parents=True, exist_ok=True)
    shutil.make_archive(str(dest/d.name),'zip',root_dir=str(d.parent),base_dir=d.name)
    print('wrote', dest/(d.name+'.zip'))
print()
print('Unzip into $SAGEGI_WORK/emb/ so that you have')
print('  $SAGEGI_WORK/emb/NT-v2-50M_t1000/*.npz')
